# 🤖 Atividade Prática — Assistente de FAQ das Aulas de PLN
### Construindo um sistema RAG com os conteúdos do próprio curso

---

## 🎯 O que você vai construir

Um **assistente inteligente** que responde perguntas sobre o conteúdo das aulas  
de PLN — usando os textos das aulas como base de conhecimento.

Você vai fazer tudo do zero, passo a passo:

```
ETAPA 1 → Preparar os textos das aulas
ETAPA 2 → Dividir em chunks (pedaços menores)
ETAPA 3 → Gerar embeddings de cada chunk
ETAPA 4 → Indexar com FAISS para busca rápida
ETAPA 5 → Buscar os chunks mais relevantes para uma pergunta
ETAPA 6 → Gerar uma resposta com um modelo de linguagem
ETAPA 7 → Reflexão e análise crítica
```

---

## 📚 O que é RAG?

**RAG** = Retrieval-Augmented Generation (Geração com Recuperação)

É uma técnica que combina dois sistemas:

```
RECUPERAÇÃO  →  busca os trechos de texto mais relevantes para a pergunta
GERAÇÃO      →  usa esses trechos como contexto para gerar uma resposta
```

Por que RAG é melhor do que só usar um LLM?

```
SEM RAG: o LLM só sabe o que aprendeu no treinamento
         → pode inventar respostas (alucinação!)

COM RAG:  o LLM recebe o trecho correto junto com a pergunta
         → responde com base em informação real e verificável
```

---

## ⏱️ Tempo estimado: 60–90 minutos

## 📦 Bibliotecas necessárias

```bash
pip install sentence-transformers faiss-cpu
```

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 1 — Instalação e imports
# Execute esta célula primeiro!
# ─────────────────────────────────────────────────────────────

# Instalar dependências (somente necessário na primeira vez)
import subprocess
subprocess.run(['pip', 'install', 'sentence-transformers', 'faiss-cpu', '-q'],
               capture_output=True)

import numpy as np
import faiss
import textwrap
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from transformers import pipeline

print('✅ Todas as bibliotecas carregadas!')
print()
print('Bibliotecas utilizadas nesta atividade:')
print('  sentence-transformers → gerar embeddings dos textos')
print('  faiss-cpu             → indexar e buscar por similaridade')
print('  transformers          → gerar respostas com LLM gratuito')

---
## 📄 ETAPA 1 — Base de Conhecimento

Esta é a **base de conhecimento** do nosso assistente —  
resumos das principais aulas do curso de PLN.

Em vez de carregar PDFs externos, usamos textos diretamente no código.  
Isso garante que a atividade funcione sem depender de arquivos externos.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 2 — Base de conhecimento: textos das aulas
#
# Cada texto representa o conteúdo de uma aula do curso.
# Em um sistema real, esses textos viriam de PDFs ou documentos.
# ─────────────────────────────────────────────────────────────

textos_aulas = {

    'aula1_representacoes': """
    Representações Conceituais em PLN. As ontologias são sistemas de organização do conhecimento
    que definem conceitos e as relações entre eles de forma hierárquica. A WordNet é um banco
    léxico do inglês que agrupa palavras por significado em conjuntos chamados synsets. Cada
    synset representa um conceito único e contém sinônimos. A BabelNet é uma versão multilíngue
    da WordNet, cobrindo mais de 500 idiomas. As representações conceituais são fundamentais
    para tarefas de desambiguação de sentido de palavras, pois uma mesma palavra pode ter
    múltiplos significados dependendo do contexto. O WordNet utiliza relações como hiponímia
    (é um tipo de), hiperonímia (é mais geral que), meronímia (é parte de) e antonímia
    (é oposto de) para estruturar o léxico.
    """,

    'aula2_ferramentas': """
    Tecnologias e Ferramentas de PLN. O NLTK é uma das bibliotecas mais antigas de PLN em
    Python, oferecendo tokenização, stemming, lematização e remoção de stopwords. O spaCy
    é uma biblioteca moderna e eficiente que processa texto com um pipeline integrado,
    oferecendo tokenização, POS tagging, reconhecimento de entidades (NER) e análise de
    dependência sintática. O Hugging Face Transformers disponibiliza milhares de modelos
    pré-treinados acessíveis via pipeline() com uma única linha de código. O TensorFlow e
    o Keras permitem construir redes neurais do zero para tarefas de PLN. O BERTimbau é o
    modelo BERT treinado especificamente em Português Brasileiro pelo grupo NeuralMind da
    UNICAMP, com 2,7 bilhões de palavras de treinamento.
    """,

    'aula3_rnn': """
    Redes Neurais Recorrentes (RNN). A RNN processa texto palavra por palavra, mantendo um
    hidden state que funciona como memória da sequência lida. O hidden state é atualizado
    a cada passo pela fórmula: ht = tanh(W * xt + U * ht-1 + b), onde xt é a palavra atual
    e ht-1 é a memória anterior. O problema do vanishing gradient ocorre quando o sinal
    de aprendizado desaparece em frases longas, fazendo a RNN esquecer o início da sequência.
    A LSTM resolve esse problema com duas memórias: o hidden state (curto prazo) e o cell
    state (longo prazo), controladas por três portões: forget gate (o que esquecer), input
    gate (o que guardar) e output gate (o que usar agora). A GRU é uma versão simplificada
    da LSTM com apenas dois portões, sendo mais rápida e eficiente.
    """,

    'aula3_transformer': """
    Arquitetura Transformer. O Transformer foi introduzido no paper Attention Is All You
    Need em 2017 por Vaswani et al. O mecanismo de atenção permite que cada token preste
    atenção em todos os outros tokens da sequência ao mesmo tempo. O Scaled Dot-Product
    Attention calcula: Attention(Q,K,V) = softmax(Q*KT / sqrt(dk)) * V, onde Q são as
    queries, K são as keys e V são os values. O Multi-Head Attention executa o mecanismo
    de atenção múltiplas vezes em paralelo, cada cabeça capturando diferentes tipos de
    relações. O Positional Encoding adiciona informação de posição ao embedding, pois o
    Transformer processa todos os tokens em paralelo sem noção de ordem. As conexões
    residuais e o Layer Normalization estabilizam o treinamento de redes profundas.
    """,

    'aula4_bert_gpt': """
    Modelos Pré-treinados BERT e GPT. O BERT usa um Encoder bidirecional que lê o texto
    nos dois sentidos simultaneamente, sendo ideal para tarefas de compreensão como
    classificação, extração e perguntas e respostas. O BERT foi pré-treinado com dois
    objetivos: Masked Language Model (MLM), onde aprende a prever palavras mascaradas, e
    Next Sentence Prediction (NSP). O GPT usa um Decoder autorregressivo que lê da esquerda
    para a direita e é treinado para prever a próxima palavra, sendo ideal para geração de
    texto. O fine-tuning é o processo de adaptar um modelo pré-treinado para uma tarefa
    específica com poucos exemplos e pouco tempo de treinamento. O DistilBERT é uma versão
    comprimida do BERT com 40% menos parâmetros mas desempenho similar.
    """,

    'aula4_bertimbau': """
    BERTimbau e Modelos para Português. O BERT original foi treinado apenas em inglês e
    tem dificuldades com Português, dividindo palavras com acento em múltiplos tokens.
    O BERTimbau foi treinado pelo NeuralMind em 2,68 bilhões de palavras do corpus brWaC,
    coletado de sites brasileiros. O modelo reconhece o vocabulário do PT-BR incluindo
    acentos, gírias e expressões idiomáticas. Para análise de sentimento em Português,
    recomenda-se o modelo lxyuan/distilbert-base-multilingual-cased-sentiments-student
    que suporta 7 idiomas incluindo Português. Para perguntas e respostas em Português,
    usa-se o pierreguillou/bert-base-cased-squad-v1.1-portuguese.
    """,

    'aula5_multimodal': """
    Aplicações Multimodais. Modelos multimodais combinam texto e imagem para criar sistemas
    mais completos. O OpenCV é usado para pré-processar imagens, convertendo de BGR para RGB
    e normalizando os pixels para o intervalo 0.0 a 1.0. O modelo BLIP da Salesforce realiza
    image captioning, gerando descrições textuais automáticas de imagens. O BLIP também
    suporta Visual Question Answering (VQA), onde responde perguntas específicas sobre
    imagens. O CLIP da OpenAI realiza classificação zero-shot de imagens, comparando
    embeddings de imagem com embeddings de texto de categorias definidas pelo usuário,
    sem necessidade de treinamento adicional. O pipeline multimodal completo combina
    pré-processamento, captioning, VQA e classificação em um único sistema.
    """,

    'aula6_traducao': """
    Tradução Automática Neural. A tradução automática evoluiu de sistemas baseados em
    regras (1950-1980) para sistemas estatísticos (1990-2014) e finalmente para modelos
    neurais baseados em Transformers (2017 em diante). A arquitetura Encoder-Decoder é
    a base dos modelos de tradução: o Encoder entende o texto de origem e o Decoder gera
    o texto de destino. O mecanismo de atenção cruzada (cross-attention) permite ao Decoder
    acessar diretamente qualquer parte do texto de origem em cada passo, eliminando o
    gargalo do vetor de contexto fixo dos modelos seq2seq. A métrica BLEU mede a precisão
    de n-gramas do texto traduzido em comparação com traduções de referência humana.
    O grupo Helsinki-NLP disponibiliza modelos de tradução para mais de 1000 pares de idiomas.
    """,

    'aula6_sumarizacao': """
    Sumarização Automática. A sumarização extrativa seleciona e copia frases do texto
    original, sendo fidedigna mas possivelmente desconectada. A sumarização abstrativa
    gera novas frases reescrevendo o conteúdo, sendo mais fluente mas podendo gerar
    alucinações, ou seja, informações não presentes no texto original. O modelo BART
    combina Encoder bidirecional e Decoder autorregressivo, sendo pré-treinado com ruído:
    aprende a reconstruir textos corrompidos por mascaramento, remoção e embaralhamento.
    A métrica ROUGE mede o recall de n-gramas do resumo gerado em relação ao resumo de
    referência. Os parâmetros max_length e min_length controlam o tamanho do resumo.
    """,

    'aula6_sentimento': """
    Análise de Sentimento. A análise de sentimento classifica textos quanto à sua polaridade
    emocional em positivo, negativo ou neutro. Existem três abordagens principais: baseada
    em léxico (usa dicionários de palavras com scores), aprendizado de máquina clássico
    (SVM, Naive Bayes com TF-IDF) e deep learning (BERT e variantes). O F1-Score Macro é
    preferido à acurácia simples para avaliar modelos em datasets desbalanceados, pois
    penaliza modelos que ignoram classes minoritárias. Desafios incluem negação (não bom
    deveria ser negativo), ironia (que serviço maravilhoso, esperei 4 horas!) e domínio
    específico. O modelo lxyuan/distilbert-multilingual suporta análise de sentimento em
    Português com três classes: positive, neutral e negative.
    """,
}

print(f'✅ Base de conhecimento criada!')
print(f'   Total de documentos: {len(textos_aulas)}')
print()
print('Documentos disponíveis:')
for nome in textos_aulas:
    n_palavras = len(textos_aulas[nome].split())
    print(f'  {nome:<30} ({n_palavras} palavras)')

---
## ✂️ ETAPA 2 — Dividir em Chunks

Modelos de embedding têm um **limite de tokens** que conseguem processar.  
Além disso, chunks menores facilitam encontrar o trecho exato relevante.

Vamos dividir cada texto em pedaços menores com **sobreposição**:

```
Texto completo:
[palavra1 palavra2 palavra3 palavra4 palavra5 palavra6 ...]

chunk_size=50, overlap=10:
Chunk 1: [palavras 1–50]
Chunk 2: [palavras 41–90]  ← sobreposição de 10 palavras
Chunk 3: [palavras 81–130] ← sobreposição de 10 palavras
```

A sobreposição garante que informações no meio de dois chunks não se percam.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 3 — Divisão em chunks com sobreposição
# ─────────────────────────────────────────────────────────────

def dividir_em_chunks(texto, nome_fonte, chunk_size=80, overlap=15):
    """
    Divide um texto em chunks (pedaços) com sobreposição.

    Parâmetros:
        texto       : texto completo a ser dividido
        nome_fonte  : identificador do documento de origem
        chunk_size  : tamanho de cada chunk em palavras
        overlap     : quantas palavras do chunk anterior repetir

    Retorna:
        lista de dicionários com 'texto' e 'fonte'
    """
    # Dividir em palavras
    palavras = texto.split()
    chunks   = []
    inicio   = 0

    while inicio < len(palavras):
        fim       = min(inicio + chunk_size, len(palavras))
        chunk_txt = ' '.join(palavras[inicio:fim])

        # Só adicionar se tiver conteúdo real
        if len(chunk_txt.strip()) > 20:
            chunks.append({
                'texto': chunk_txt,
                'fonte': nome_fonte,
                'indice_chunk': len(chunks)
            })

        # Avançar com sobreposição
        inicio += chunk_size - overlap

    return chunks


# Aplicar a todos os textos das aulas
todos_chunks = []

for nome, texto in textos_aulas.items():
    chunks_doc = dividir_em_chunks(texto, nome_fonte=nome)
    todos_chunks.extend(chunks_doc)

print(f'✅ Chunking concluído!')
print(f'   Total de chunks gerados: {len(todos_chunks)}')
print(f'   Documentos originais   : {len(textos_aulas)}')
print(f'   Média de chunks/doc    : {len(todos_chunks)/len(textos_aulas):.1f}')
print()
print('Primeiros 3 chunks (para verificação):')
print('─' * 55)
for chunk in todos_chunks[:3]:
    print(f'  Fonte : {chunk["fonte"]}')
    print(f'  Texto : "{chunk["texto"][:100]}..."')
    print(f'  Palavras: {len(chunk["texto"].split())}')
    print()

---
## 🔢 ETAPA 3 — Gerar Embeddings

Um **embedding** é a representação de um texto como um vetor de números.  
Textos com significados parecidos ficam com vetores parecidos.

```
"hidden state da RNN"  →  [0.32, -0.18, 0.74, 0.51, ...] (384 números)
"memória da rede RNN"  →  [0.29, -0.21, 0.71, 0.48, ...] (muito similar!)
"receita de bolo"      →  [-0.8, 0.60, -0.3, 0.12, ...] (muito diferente)
```

Usamos o modelo `all-MiniLM-L6-v2` — pequeno (~90 MB), rápido e multilíngue.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 4 — Carregar modelo e gerar embeddings
# ─────────────────────────────────────────────────────────────

print('🔄 Carregando modelo de embeddings...')
modelo_embedding = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Modelo carregado!')
print(f'   Dimensão dos embeddings: {modelo_embedding.get_sentence_embedding_dimension()}')
print()

# Extrair apenas os textos para gerar os embeddings
textos_chunks = [c['texto'] for c in todos_chunks]

print(f'🔄 Gerando embeddings para {len(textos_chunks)} chunks...')
embeddings = modelo_embedding.encode(
    textos_chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)
print()
print(f'✅ Embeddings gerados!')
print(f'   Shape da matriz: {embeddings.shape}')
print(f'   Interpretação  : {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensões')
print()

# Verificar que embeddings similares ficam próximos
from sklearn.metrics.pairwise import cosine_similarity

frases_teste_sim = [
    'O que é o hidden state da RNN?',
    'Como funciona a memória da rede recorrente?',
    'Qual é a diferença entre BERT e GPT?',
]
emb_teste = modelo_embedding.encode(frases_teste_sim)
sims      = cosine_similarity(emb_teste)

print('🔍 Verificação de similaridade entre frases:')
print(f'   Frase 1 ↔ Frase 2 (mesma ideia): {sims[0,1]:.3f}  (esperado: alto)')
print(f'   Frase 1 ↔ Frase 3 (ideia difer.): {sims[0,2]:.3f}  (esperado: baixo)')

---
## 🗂️ ETAPA 4 — Indexar com FAISS

O **FAISS** (Facebook AI Similarity Search) é uma biblioteca de busca vetorial  
ultra-rápida. Em vez de comparar a pergunta com todos os chunks um por um,  
o FAISS usa estruturas de dados especiais para encontrar os mais similares  
em milissegundos — mesmo com milhões de vetores.

```
Busca normal: comparar com TODOS os chunks → lenta
FAISS:        estrutura especial de índice → muito rápida
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 5 — Criar índice FAISS
# ─────────────────────────────────────────────────────────────

# Dimensão dos vetores (determinada pelo modelo de embedding)
DIM = embeddings.shape[1]

# Normalizar embeddings para busca por similaridade de cosseno
# (necessário para que IndexFlatIP funcione como cosseno)
embeddings_norm = embeddings.copy().astype('float32')
faiss.normalize_L2(embeddings_norm)  # normaliza cada vetor para norma = 1

# Criar índice FAISS
# IndexFlatIP = Inner Product (produto escalar = cosseno para vetores normalizados)
indice = faiss.IndexFlatIP(DIM)

# Adicionar os embeddings ao índice
indice.add(embeddings_norm)

print('✅ Índice FAISS criado!')
print(f'   Vetores indexados: {indice.ntotal}')
print(f'   Dimensão         : {DIM}')
print()
print('💡 O FAISS permite buscar o chunk mais relevante')
print('   entre milhares de vetores em milissegundos!')

---
## 🔍 ETAPA 5 — Busca Semântica

Agora vamos implementar a **recuperação**: dado uma pergunta do usuário,  
encontrar os chunks mais relevantes.

```
Pergunta: "O que é o hidden state?"
      ↓
Embedding da pergunta: [0.31, -0.20, 0.73, ...]
      ↓
FAISS busca os N vetores mais próximos
      ↓
Retorna: chunk da aula3_rnn (score: 0.91) ✅
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 6 — Função de busca semântica
# ─────────────────────────────────────────────────────────────

def buscar_chunks(pergunta, top_k=3):
    """
    Busca os chunks mais relevantes para uma pergunta.

    Parâmetros:
        pergunta : pergunta do usuário (string)
        top_k    : quantos chunks retornar

    Retorna:
        lista de dicionários com chunk, score e posição
    """
    # Gerar embedding da pergunta
    emb_pergunta = modelo_embedding.encode([pergunta],
                                            convert_to_numpy=True).astype('float32')

    # Normalizar (igual ao que fizemos com os chunks)
    faiss.normalize_L2(emb_pergunta)

    # Buscar os top_k mais próximos
    scores, indices = indice.search(emb_pergunta, top_k)

    # Montar lista de resultados
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:  # -1 indica resultado inválido
            resultados.append({
                'chunk'  : todos_chunks[idx],
                'score'  : float(score),
                'posicao': int(idx)
            })

    return resultados


# Testar a busca com algumas perguntas
perguntas_teste = [
    'O que é o hidden state de uma RNN?',
    'Qual a diferença entre BERT e GPT?',
    'Para que serve a função softmax na atenção?',
    'O que é fine-tuning de modelos pré-treinados?',
    'Como funciona o FAISS para busca por similaridade?',
]

print('🔍 Testando a busca semântica:\n')
print('=' * 60)

for pergunta in perguntas_teste:
    resultados = buscar_chunks(pergunta, top_k=3)
    print(f'\n❓ Pergunta: "{pergunta}"')
    print(f'   Chunks encontrados:')
    for i, res in enumerate(resultados):
        fonte = res['chunk']['fonte']
        score = res['score']
        texto = res['chunk']['texto'][:80] + '...'
        print(f'   {i+1}. [{fonte}] score={score:.3f}')
        print(f'      "{texto}"')

---
## 💬 ETAPA 6 — Gerar Resposta com LLM

Com os chunks relevantes em mãos, montamos um **prompt de contexto**  
e pedimos ao modelo que gere uma resposta baseada nele.

```
PROMPT MONTADO:
  Contexto 1: [texto do chunk mais relevante]
  Contexto 2: [texto do segundo chunk]
  Contexto 3: [texto do terceiro chunk]
  
  Pergunta: O que é o hidden state?
  Resposta:
```

Usamos o **Flan-T5** — modelo gratuito, sem chave de API, que roda localmente.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 7 — Carregar o modelo de geração de respostas
#
# google/flan-t5-base
#   → Modelo gratuito e de código aberto
#   → Não precisa de API key
#   → Treinado para seguir instruções em inglês
#   → ~250 MB — muito menor que BERT ou GPT-2
# ─────────────────────────────────────────────────────────────

print('🔄 Carregando modelo de geração (Flan-T5)...')
gerador = pipeline(
    'text2text-generation',
    model='google/flan-t5-base'
)
print('✅ Modelo carregado!')
print()
print('💡 O Flan-T5 foi treinado para seguir instruções.')
print('   É menor e mais rápido que o GPT — perfeito para esta atividade.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 8 — Função RAG completa: busca + geração
# ─────────────────────────────────────────────────────────────

def montar_prompt(pergunta, chunks_relevantes):
    """
    Monta o prompt com o contexto dos chunks recuperados.

    O prompt instrui o modelo a responder baseado apenas
    no contexto fornecido — evitando alucinações.
    """
    contexto = ''
    for i, res in enumerate(chunks_relevantes):
        texto = res['chunk']['texto']
        fonte = res['chunk']['fonte']
        contexto += f'Context {i+1} [{fonte}]: {texto}\n\n'

    prompt = (
        f'Based on the following context, answer the question clearly and concisely.\n\n'
        f'{contexto}'
        f'Question: {pergunta}\n'
        f'Answer:'
    )
    return prompt


def responder_com_rag(pergunta, top_k=3, max_tokens=200):
    """
    Pipeline RAG completo:
        1. Busca os chunks mais relevantes
        2. Monta o prompt com contexto
        3. Gera a resposta com o LLM

    Retorna:
        dicionário com resposta, chunks usados e scores
    """
    # Passo 1: Recuperar chunks relevantes
    chunks_relevantes = buscar_chunks(pergunta, top_k=top_k)

    # Passo 2: Montar prompt
    prompt = montar_prompt(pergunta, chunks_relevantes)

    # Passo 3: Gerar resposta
    resultado = gerador(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=False  # determinístico para reprodutibilidade
    )
    resposta = resultado[0]['generated_text']

    return {
        'pergunta'         : pergunta,
        'resposta'         : resposta,
        'chunks_usados'    : chunks_relevantes,
        'fontes'           : [r['chunk']['fonte'] for r in chunks_relevantes],
        'scores'           : [r['score'] for r in chunks_relevantes],
    }


# Testar o pipeline RAG completo
perguntas_rag = [
    'What is the hidden state in a RNN?',
    'What is the difference between BERT and GPT?',
    'How does the attention mechanism work in Transformers?',
    'What is fine-tuning and why is it useful?',
    'What is sentiment analysis and what are its challenges?',
]

print('🤖 Pipeline RAG Completo — Perguntas e Respostas\n')
print('=' * 60)

for pergunta in perguntas_rag:
    resultado = responder_com_rag(pergunta, top_k=3)

    print(f'\n❓ Pergunta:')
    print(f'   {pergunta}')
    print(f'\n🤖 Resposta:')
    for linha in textwrap.wrap(resultado['resposta'], width=58):
        print(f'   {linha}')
    print(f'\n📚 Fontes usadas:')
    for fonte, score in zip(resultado['fontes'], resultado['scores']):
        print(f'   [{fonte}] (score: {score:.3f})')
    print('─' * 60)

---
## 📊 ETAPA 7 — Análise e Avaliação do Sistema

Um bom sistema RAG não é só implementado — ele é **avaliado e compreendido**.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 9 — Visualização dos scores de busca
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt

perguntas_viz = [
    'What is the hidden state?',
    'How does BERT work?',
    'What is sentiment analysis?',
]

fig, axes = plt.subplots(1, len(perguntas_viz), figsize=(14, 5))

for ax, pergunta in zip(axes, perguntas_viz):
    resultados = buscar_chunks(pergunta, top_k=5)

    fontes = [r['chunk']['fonte'].replace('aula', 'A').replace('_', '\n')[:12]
              for r in resultados]
    scores = [r['score'] for r in resultados]

    bars = ax.barh(fontes[::-1], scores[::-1],
                   color='#534AB7', alpha=0.85, edgecolor='white')
    ax.set_title(f'"{pergunta[:30]}"', fontsize=9, fontweight='bold')
    ax.set_xlabel('Score de similaridade')
    ax.set_xlim(0, 1)
    ax.grid(alpha=0.3, axis='x')

    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.01,
                bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=8)

plt.suptitle('Scores de Similaridade — FAISS retrieval por pergunta',
             fontweight='bold', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('💡 Scores próximos de 1.0 = chunk muito relevante para a pergunta')
print('   Scores abaixo de 0.3   = pouca relação semântica')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 10 — Comparativo: top_k diferente
# Veja como o número de chunks afeta a qualidade da resposta
# ─────────────────────────────────────────────────────────────

pergunta_comp = 'What is the difference between RNN and LSTM?'

print(f'📌 Pergunta: "{pergunta_comp}"\n')
print('=' * 60)

for k in [1, 2, 3]:
    resultado = responder_com_rag(pergunta_comp, top_k=k)
    print(f'\n  top_k = {k}:')
    print(f'  Fontes: {resultado["fontes"]}')
    print(f'  Resposta:')
    for linha in textwrap.wrap(resultado['resposta'], width=55):
        print(f'    {linha}')

print()
print('🤔 Reflexão: como o top_k afeta a qualidade da resposta?')

---
## ✍️ ETAPA 8 — Reflexão Final (Obrigatória)

Responda as perguntas abaixo **nas células de texto** (dê duplo clique para editar).  
Esta etapa vale **30% da nota da atividade**.

---

### ❓ Pergunta 1

**O que acontece com a qualidade das respostas se você diminuir o `top_k` de 3 para 1?  
Por que o número de chunks recuperados importa?**

✏️ *Sua resposta aqui:*

___


### ❓ Pergunta 2

**Por que usar embeddings para busca é melhor do que simplesmente procurar  
palavras-chave no texto? Dê um exemplo de pergunta onde a busca por  
palavra-chave falharia mas a busca semântica funcionaria.**

✏️ *Sua resposta aqui:*

___


### ❓ Pergunta 3

**Qual seria a principal limitação deste sistema se usado em produção  
com documentos reais muito longos (ex: um livro inteiro)?  
Como você resolveria essa limitação?**

✏️ *Sua resposta aqui:*

___


### ❓ Pergunta 4 (Desafio)

**O sistema atual não tem memória de conversa — cada pergunta é independente.  
Como você modificaria o código para que o assistente lembre das  
perguntas anteriores da sessão?**

✏️ *Sua resposta aqui (pode ser em pseudocódigo):*

___


---
## 🏆 Desafios Opcionais (Bônus)

Escolha **pelo menos um** para ganhar pontos extras:

| Desafio | Bônus | Descrição |
|---|---|---|
| 🟡 Básico | +0.5 pt | Mude o `chunk_size` de 80 para 50. Quantos chunks são gerados? A qualidade melhora ou piora? |
| 🟠 Médio | +1.0 pt | Adicione um novo texto ao dicionário `textos_aulas` sobre um tema de PLN que você escolher e verifique se o sistema consegue respondê-lo |
| 🔴 Avançado | +2.0 pt | Implemente um **histórico de sessão**: armazene as últimas 3 perguntas e respostas e inclua-as no prompt como contexto de conversa |

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA BÔNUS — Espaço para os desafios opcionais
# ─────────────────────────────────────────────────────────────

# Seu código aqui!


---
## 📝 Resumo do que você construiu

```
┌──────────────────────────────────────────────────────────┐
│              PIPELINE RAG — ASSISTENTE DE PLN             │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  📄 Textos das aulas  ──→  chunks  ──→  embeddings       │
│                                          (offline)       │
│                              ↓                           │
│                         Índice FAISS                     │
│                                                          │
│  ❓ Pergunta ──→  embedding  ──→  busca FAISS             │
│                                       ↓                  │
│                           top-k chunks relevantes        │
│                                       ↓                  │
│                     contexto + pergunta → Flan-T5        │
│                                       ↓                  │
│                         ✅ Resposta fundamentada          │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

### ✅ Conceitos aprendidos nesta atividade

- O que é RAG e por que ele reduz alucinações
- Como dividir textos em chunks com sobreposição
- O que são embeddings e como comparar textos semanticamente
- Como usar FAISS para busca vetorial eficiente
- Como montar um prompt de contexto para guiar um LLM
- O impacto do parâmetro `top_k` na qualidade das respostas
- A diferença entre busca por palavra-chave e busca semântica